# 05 · Топ-каналы по «шуму» (частоте событий)

**Цель** — найти каналы с максимальной частотой событий («шум»): для задачи «отказ датчика» паттерны ложных сработок и частоты тревог — ключевые признаки деградации.

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import data_utils as du

print("dataset:", du.find_dataset_dir())

# 5.1 · Топ-20 шумных каналов за 2025 (артефакт `top_channels_2025.csv`)

Берём свежий полный год — 2025.

In [ ]:
out_csv = du.find_dataset_dir() / "top_channels_2025.csv"
year = "2025"
fp = next(f for f in du.journal_files() if year in f.name)
counts = {}
for ch in du.read_chunks(fp, cols=["ид_канала_данных", "тревожное"]):
    counts.update(ch["ид_канала_данных"].value_counts().to_dict())
top = pd.Series(counts).sort_values(ascending=False)
top.rename("событий").to_csv(out_csv, encoding="utf-8-sig")

ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "название_датчика"]]
top_df = (top.head(20).rename("событий").to_frame()
          .reset_index().rename(columns={"index": "ид_канала_данных"}))
top_df = top_df.merge(ref_ch, on="ид_канала_данных", how="left")
print(f"Топ-20 шумных каналов ({year}):")
print(top_df.to_string(index=False))

# 5.2 · Комментарии

- Почти весь топ-шум — **газовые датчики** (60–65 тыс. событий/год на канал). Их «нормальное» поведение — частая телеметрия; отклонение от индивидуального фона — признак деградации.
- Каналы телеметрии `0.00/0.01` генерируют тысячи событий в день — для них важна **нормировка частоты по каналу** (z-score по историческому фону).
- Отсюда признак «интенсивность событий» в будущем feature pipeline обязан быть **относительным** (доля от медианы канала), а не абсолютным.